In [ ]:
!apt-get update -qq && apt-get install -y -qq zstd
!pip -q install google-genai requests python-dotenv pytest
!curl -fsSL https://ollama.com/install.sh | sh

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
import subprocess, time

# Start Ollama server in the background
subprocess.Popen(["ollama", "serve"])
time.sleep(5)  # Wait for server to initialize

# Check version and pull llama3.2 model
!ollama --version
!ollama pull llama3.2

ollama version is 0.40.16



In [ ]:
%%writefile llm_client.py
from abc import ABC, abstractmethod
from google import genai
import requests

# Common interface
class LLMProvider(ABC):
    @abstractmethod
    def generate(self, prompt):
        pass

# Gemini cloud provider
class GeminiProvider(LLMProvider):
    def __init__(self, api_key):
        self.client = genai.Client(api_key=api_key)

    def generate(self, prompt):
        response = self.client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt
        )
        return response.text

# Ollama local provider
class OllamaProvider(LLMProvider):
    def __init__(self, model="llama3.2"):
        self.model = model
        self.url = "http://localhost:11434/api/generate"

    def generate(self, prompt):
        response = requests.post(
            self.url,
            json={
                "model": self.model,
                "prompt": prompt,
                "stream": False
            },
            timeout=120
        )
        response.raise_for_status()
        return response.json()["response"]

# Common client
class LLMClient:
    def __init__(self, provider):
        self.provider = provider

    def generate(self, prompt):
        return self.provider.generate(prompt)

Overwriting llm_client.py


In [ ]:
%%writefile test_client.py
import unittest
from llm_client import LLMClient

class FakeProvider:
    def generate(self, prompt):
        return "AI stands for Artificial Intelligence."

class TestLLMClient(unittest.TestCase):
    def test_generate(self):
        provider = FakeProvider()
        client = LLMClient(provider)
        result = client.generate("Explain AI")
        self.assertEqual(
            result,
            "AI stands for Artificial Intelligence."
        )

if __name__ == "__main__":
    unittest.main()

Writing test_client.py


In [ ]:
!python test_client.py

.
----------------------------------------------------------------------
Ran 1 test in 0.000s

OK


In [ ]:
import os
from google.colab import userdata
from llm_client import GeminiProvider, OllamaProvider, LLMClient

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

def run_client():
    provider_name = input("Choose provider (gemini/ollama): ").strip().lower()
    if provider_name == "gemini":
        api_key = os.getenv("GEMINI_API_KEY")
        if not api_key:
            raise ValueError("Gemini API key not found")
        provider = GeminiProvider(api_key)
    elif provider_name == "ollama":
        provider = OllamaProvider("llama3.2")
    else:
        raise ValueError("Invalid provider selected")

    client = LLMClient(provider)
    prompt = input("Enter your question: ")
    answer = client.generate(prompt)
    print("\nLLM Response:")
    print(answer)

run_client()

Choose provider (gemini/ollama): gemini
Enter your question: what is machine learning

LLM Response:
Machine Learning (ML) is a field of Artificial Intelligence (AI) that enables systems to **learn from data, identify patterns, and make decisions or predictions with minimal human intervention**, rather than being explicitly programmed for each task.

Think of it this way:

*   **Traditional Programming:** You, the programmer, write specific rules for every possible scenario. If X happens, do Y. If Z happens, do W. The computer just follows these rules.
*   **Machine Learning:** You, the developer, don't write rules for every scenario. Instead, you give the computer a lot of data and an algorithm (a set of instructions on how to learn). The computer then analyzes that data to **learn its own rules or patterns**. Once it has learned, it can apply those learned patterns to new, unseen data to make predictions or decisions.

**Here's a breakdown of the core concepts:**

1.  **Learning from